# 04 – Fine-Tuning SigLIP SO400M (Phase 5)

**NEEDS GPU T4 x2.**

Current best to beat: **val R-prec = 0.9287** (WiSE-FT α=0.3, LB=0.9511).

Recipe (AGENTS.md Phase 5):
- Stage A: freeze backbone, train linear head only (5 epochs, LR=1e-3)
- Stage B: unfreeze last 4 transformer blocks with LLRD (15 epochs)
- AMP fp16, EMA decay=0.9999, cosine LR schedule, label smoothing=0.1
- Early stopping patience=5 on val R-precision
- Final: WiSE-FT blend fine-tuned logits + zero-shot logits, tune α on val

In [ ]:
import sys, pathlib
sys.path.insert(0, "/kaggle/input/datasets/somyajitsaha/food-retrieval-src")
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from src.config import seed_everything
seed_everything(42)

try:
    import open_clip
except ImportError:
    import subprocess, sys as _s
    subprocess.run([_s.executable, "-m", "pip", "install", "open-clip-torch", "-q"])
    import open_clip

import torchvision, timm, sklearn
print({k: v.__version__ for k, v in dict(
    torch=torch, torchvision=torchvision, timm=timm,
    sklearn=sklearn, numpy=np).items()})
print("open_clip:", open_clip.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1),
      "GB" if torch.cuda.is_available() else "")

DATA_DIR = pathlib.Path("/kaggle/input/competitions/match-the-dish-to-its-rizz")
OUT_DIR  = pathlib.Path("/kaggle/working")
CKPT_DIR = OUT_DIR / "checkpoints"; CKPT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "submissions").mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DATA_DIR exists:", DATA_DIR.exists())

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
from src.data import load_csvs, parse_val_ground_truth, GalleryDataset, get_transforms
from src.text_matching import build_surface_form_dict, TextMatcher
from torch.utils.data import DataLoader, Dataset
from PIL import Image

dfs          = load_csvs(DATA_DIR)
train_df     = dfs["train"]
val_gallery  = dfs["val_gallery"]
val_queries  = dfs["val_queries"]
val_gt_df    = dfs["val_gt"]
test_gallery = dfs["test_gallery"]
test_queries = dfs["test_queries"]
meta         = dfs["category_metadata"]
val_gt       = parse_val_ground_truth(val_gt_df)

cat_id_list = meta["category_id"].tolist()
cat_id2idx  = {c: i for i, c in enumerate(cat_id_list)}
K           = len(cat_id_list)
matcher     = TextMatcher(build_surface_form_dict(meta), meta_df=meta)
print(f"K={K}  train={len(train_df)}  val_gallery={len(val_gallery)}")

In [ ]:
# ── Extract / reload SigLIP features if not cached ────────────────────────────
from src.features import (
    extract_image_features, save_features, load_features,
    extract_text_features, aggregate_text_features, save_text_features, load_text_features,
    build_prompt_texts
)
import ast

feat_dir = OUT_DIR / 'features' / 'siglip_so400m'
feat_dir.mkdir(parents=True, exist_ok=True)

need_image = not all((feat_dir / f'{s}_image.npy').exists() for s in ['train','val','test'])
need_text  = not (feat_dir / 'text.npy').exists()

if need_image or need_text:
    print('Loading SigLIP for feature extraction...')
    _model, _, _preprocess = open_clip.create_model_and_transforms(
        'ViT-SO400M-14-SigLIP', pretrained='webli')
    _model = _model.to(DEVICE).eval()

if need_image:
    print('Extracting image features...')
    for split, df, img_dir in [
        ('train', train_df,    DATA_DIR/'train_gallery'),
        ('val',   val_gallery, DATA_DIR/'validation_gallery'),
        ('test',  test_gallery,DATA_DIR/'test_gallery'),
    ]:
        if (feat_dir / f'{split}_image.npy').exists():
            print(f'  {split}: cache hit'); continue
        ds     = GalleryDataset(df, img_dir, transform=_preprocess)
        loader = DataLoader(ds, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)
        feats, ids = extract_image_features(_model, loader, device=DEVICE)
        save_features(feats, ids, feat_dir, split)
    print('Image features done')

if need_text:
    print('Extracting text features...')
    _tokenize  = open_clip.get_tokenizer('ViT-SO400M-14-SigLIP')
    cat_names  = meta['category_name'].tolist()
    alt_dict   = {}
    for idx, row in meta.reset_index(drop=True).iterrows():
        raw = row.get('alternative_names', '')
        if isinstance(raw, str) and raw.strip():
            try:    alt_dict[idx] = ast.literal_eval(raw)
            except: alt_dict[idx] = [s.strip() for s in raw.split(',') if s.strip()]
    desc_dict  = {i: str(r['description']) for i, r in meta.reset_index(drop=True).iterrows()}
    texts, cat_indices = build_prompt_texts(cat_names, alternative_names=alt_dict, descriptions=desc_dict)
    def _encode(batch):
        tokens = _tokenize(batch).to(DEVICE)
        with torch.no_grad():
            emb = _model.encode_text(tokens)
        return emb.float().cpu().numpy()
    raw_feats = extract_text_features(_encode, texts, batch_size=256)
    cat_proto = aggregate_text_features(raw_feats, cat_indices, K)
    save_text_features(cat_proto, list(range(K)), feat_dir, tag='text')
    print('Text features done')

if need_image or need_text:
    del _model; torch.cuda.empty_cache()
    print('Feature extraction complete, VRAM freed')
else:
    print('All features cached, skipping extraction')

# Verify
T_proto_check, _ = load_text_features(feat_dir, tag='text')
print(f'Text prototypes shape: {T_proto_check.shape}')


In [ ]:
# ── Classification dataset ────────────────────────────────────────────────────
# SigLIP uses mean=0.5, std=0.5
MEAN, STD, IMG_SIZE = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5), 224

tf_train = get_transforms("train", size=IMG_SIZE, mean=MEAN, std=STD)
tf_val   = get_transforms("val",   size=IMG_SIZE, mean=MEAN, std=STD)

class FoodClassDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df      = df.reset_index(drop=True)
        self.img_dir = pathlib.Path(img_dir)
        self.tf      = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row   = self.df.iloc[i]
        img   = Image.open(self.img_dir/f"{row['image_id']}.jpg").convert("RGB")
        label = cat_id2idx[row["category_id"]]
        return self.tf(img), label

train_ds = FoodClassDataset(train_df, DATA_DIR/"train_gallery", tf_train)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,
                           num_workers=4, pin_memory=True, persistent_workers=True)
print(f"Train batches: {len(train_loader)}")

In [ ]:
# ── Load SigLIP backbone + text-initialised linear head ───────────────────────
from src.features import load_text_features

backbone, _, _ = open_clip.create_model_and_transforms(
    "ViT-SO400M-14-SigLIP", pretrained="webli")
backbone = backbone.to(DEVICE)

with torch.no_grad():
    dummy  = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    emb_dim = backbone.encode_image(dummy).shape[-1]
print(f"Embedding dim: {emb_dim}")

# Linear head initialised from zero-shot text weights
feat_dir = OUT_DIR / "features" / "siglip_so400m"
T_proto, _ = load_text_features(feat_dir, tag="text")   # (K, D)

head = nn.Linear(emb_dim, K, bias=False).to(DEVICE)
with torch.no_grad():
    head.weight.copy_(torch.tensor(T_proto, dtype=torch.float32).to(DEVICE))
print("Head initialised from zero-shot text weights")

In [ ]:
# ── Shared training utilities ─────────────────────────────────────────────────
from src.train import EMA, cosine_lr_at, get_criterion
from src.features import load_features
from src.ensemble import temperature_scale, blend_zeroshot_finetuned
from src.metrics import evaluate, print_metrics
import math

criterion = get_criterion(K, label_smoothing=0.1)
scaler    = torch.cuda.amp.GradScaler()

# Load cached val/test features for fast eval (no re-extraction each epoch)
X_val,  val_ids  = load_features(feat_dir, "val",  gallery_df=val_gallery)
X_test, test_ids = load_features(feat_dir, "test", gallery_df=test_gallery)

# Zero-shot val scores (reference)
S_zs_val  = X_val  @ T_proto.T
S_zs_test = X_test @ T_proto.T

def score_to_preds(S, gallery_ids, query_df):
    pred_dict = {}
    for _, row in query_df.iterrows():
        cat_id, _, _ = matcher.predict(str(row["description"]))
        cat_idx = cat_id2idx.get(cat_id, 0)
        order   = np.argsort(-S[:, cat_idx], kind="stable")
        pred_dict[str(row["query_id"])] = [gallery_ids[j] for j in order[:20]]
    return pred_dict

def eval_head_on_cached(head_weights, X, gallery_ids, query_df, gt, alpha=0.0):
    """Score = WiSE-FT blend of cached features × head weights + zero-shot."""
    W       = head_weights.detach().cpu().numpy()          # (K, D)
    S_ft    = temperature_scale(X @ W.T, T=1.0, to_prob=True)
    if alpha > 0:
        S_zs = temperature_scale(X @ T_proto.T, T=1.0, to_prob=True)
        S    = (1 - alpha) * S_zs + alpha * S_ft
    else:
        S = S_ft
    preds   = score_to_preds(S, gallery_ids, query_df)
    return evaluate(preds, gt)["r_precision"]

print("Utilities ready")

In [ ]:
# ── STAGE A: head only (backbone frozen) ─────────────────────────────────────
EPOCHS_A = 5
LR_A     = 1e-3
WD       = 0.05

for p in backbone.parameters(): p.requires_grad_(False)
head.requires_grad_(True)

opt_a      = torch.optim.AdamW(head.parameters(), lr=LR_A, weight_decay=WD)
ema        = EMA(head, decay=0.9999)
total_a    = EPOCHS_A * len(train_loader)
warmup_a   = int(0.05 * total_a)
best_rp_a  = -1.0

print(f"Stage A: {EPOCHS_A} epochs  head-only  LR={LR_A}")
backbone.eval()

for epoch in range(1, EPOCHS_A + 1):
    head.train(); tot_loss = tot_correct = tot_n = 0
    for step, (imgs, labels) in enumerate(train_loader):
        gs  = (epoch - 1) * len(train_loader) + step
        lr  = cosine_lr_at(gs, total_a, warmup_a, LR_A)
        for pg in opt_a.param_groups: pg["lr"] = lr
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        opt_a.zero_grad()
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            with torch.no_grad():
                f = F.normalize(backbone.encode_image(imgs).float(), dim=-1)
            logits = head(f)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(head.parameters(), 1.0)
        scaler.step(opt_a); scaler.update()
        ema.update(head)
        tot_loss += loss.item(); tot_correct += (logits.argmax(1)==labels).sum().item()
        tot_n    += labels.size(0)

    # Eval with EMA weights on cached val features
    ema.apply(head)
    rp = eval_head_on_cached(head.weight, X_val, val_ids, val_queries, val_gt, alpha=0.0)
    ema.restore(head)

    if rp > best_rp_a:
        best_rp_a = rp
        ema.apply(head)
        torch.save({"head": head.state_dict(), "epoch": epoch},
                   CKPT_DIR / "stage_a_best.pt")
        ema.restore(head)

    print(f"  Epoch {epoch}/{EPOCHS_A}  loss={tot_loss/len(train_loader):.4f}  "
          f"train_acc={tot_correct/tot_n:.4f}  val_R-prec={rp:.4f}  lr={lr:.2e}")

# Load best Stage A head
ckpt = torch.load(CKPT_DIR / "stage_a_best.pt")
head.load_state_dict(ckpt["head"])
print(f"\nStage A best val R-prec: {best_rp_a:.4f}  (epoch {ckpt['epoch']})")

In [ ]:
# ── Free Stage A VRAM before Stage B ──────────────────────────────────────────
import gc, os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
try:
    del backbone, head, opt_a, ema, scaler
except Exception as e:
    print(f'Note: {e}')
gc.collect(); torch.cuda.empty_cache()
free_gb = torch.cuda.mem_get_info()[0] / 1e9
print(f'VRAM free after cleanup: {free_gb:.1f} GB')


In [ ]:
# ── Reload backbone + Stage A head for Stage B ────────────────────────────────
import open_clip, torch.nn as nn, torch.nn.functional as F
from src.features import load_text_features

backbone, _, _ = open_clip.create_model_and_transforms(
    'ViT-SO400M-14-SigLIP', pretrained='webli')
backbone = backbone.to(DEVICE)

feat_dir = OUT_DIR / 'features' / 'siglip_so400m'
T_proto, _ = load_text_features(feat_dir, tag='text')
emb_dim = T_proto.shape[1]

head = nn.Linear(emb_dim, K, bias=False).to(DEVICE)
ckpt = torch.load(CKPT_DIR / 'stage_a_best.pt')
head.load_state_dict(ckpt['head'])
print(f'Loaded Stage A best (epoch {ckpt["epoch"]})')

scaler = torch.cuda.amp.GradScaler()
used_gb = (torch.cuda.mem_get_info()[1] - torch.cuda.mem_get_info()[0]) / 1e9
print(f'VRAM used after reload: {used_gb:.1f} GB')


In [ ]:
# ── STAGE B: unfreeze last 4 blocks with LLRD ─────────────────────────────────────────
from src.models import param_groups_lrd
from src.train import EMA, cosine_lr_at, get_criterion
from src.features import load_features
from src.ensemble import temperature_scale
from src.metrics import evaluate

EPOCHS_B    = 15
BACKBONE_LR = 1e-5
HEAD_LR_B   = 1e-3
N_UNFREEZE  = 4
PATIENCE    = 5
BATCH_B     = 8     # small batch for VRAM
ACCUM_STEPS = 8     # effective batch = 64
WD          = 0.05

# Reload cached val features
X_val,  val_ids  = load_features(feat_dir, 'val',  gallery_df=val_gallery)
X_test, test_ids = load_features(feat_dir, 'test', gallery_df=test_gallery)

def eval_head_on_cached(head_weights, X, gallery_ids, query_df, gt, alpha=0.0):
    W    = head_weights.detach().cpu().numpy()
    S_ft = temperature_scale(X @ W.T, T=1.0, to_prob=True)
    if alpha > 0:
        S_zs = temperature_scale(X @ T_proto.T, T=1.0, to_prob=True)
        S    = (1 - alpha) * S_zs + alpha * S_ft
    else:
        S = S_ft
    pred_dict = {}
    for _, row in query_df.iterrows():
        cat_id, _, _ = matcher.predict(str(row['description']))
        cat_idx = cat_id2idx.get(cat_id, 0)
        order   = np.argsort(-S[:, cat_idx], kind='stable')
        pred_dict[str(row['query_id'])] = [gallery_ids[j] for j in order[:20]]
    return evaluate(pred_dict, gt)['r_precision']

train_loader_b = DataLoader(train_ds, batch_size=BATCH_B, shuffle=True,
                             num_workers=2, pin_memory=True)

# Gradient checkpointing
try:
    backbone.set_grad_checkpointing(enable=True)
    print('Gradient checkpointing enabled')
except Exception as e:
    print(f'Grad checkpointing: {e}')

# Count and unfreeze last N blocks
all_block_names = sorted(set(
    int(n.split('visual.transformer.resblocks.')[1].split('.')[0])
    for n, _ in backbone.named_parameters()
    if 'visual.transformer.resblocks.' in n
))
n_blocks = max(all_block_names) + 1 if all_block_names else 27
print(f'Blocks: {n_blocks}  unfreezing last {N_UNFREEZE}')

for name, p in backbone.named_parameters():
    block_num = -1
    if 'visual.transformer.resblocks.' in name:
        try: block_num = int(name.split('visual.transformer.resblocks.')[1].split('.')[0])
        except: pass
    p.requires_grad_(block_num >= n_blocks - N_UNFREEZE)

pg = param_groups_lrd(backbone, base_lr=BACKBONE_LR, head_lr=HEAD_LR_B,
                       wd=WD, decay=0.75, n_layers=n_blocks)
pg.append({'params': list(head.parameters()), 'lr': HEAD_LR_B, 'weight_decay': WD})

opt_b    = torch.optim.AdamW(pg, betas=(0.9, 0.999), eps=1e-8)
ema_b    = EMA(head, decay=0.9999)
crit_b   = get_criterion(K, label_smoothing=0.1)
total_b  = EPOCHS_B * len(train_loader_b)
warmup_b = int(0.05 * total_b)
best_rp_b, no_improve = -1.0, 0

trainable  = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
trainable += sum(p.numel() for p in head.parameters())
print(f'Trainable: {trainable/1e6:.1f}M  batch={BATCH_B}x{ACCUM_STEPS}={BATCH_B*ACCUM_STEPS}')
torch.cuda.empty_cache()

for epoch in range(1, EPOCHS_B + 1):
    backbone.train(); head.train()
    tot_loss = tot_correct = tot_n = 0
    opt_b.zero_grad()
    for step, (imgs, labels) in enumerate(train_loader_b):
        gs  = (epoch - 1) * len(train_loader_b) + step
        lr  = cosine_lr_at(gs, total_b, warmup_b, HEAD_LR_B)
        for pg_i in opt_b.param_groups:
            ratio = pg_i['lr'] / HEAD_LR_B if HEAD_LR_B > 0 else 1.0
            pg_i['lr'] = lr * ratio
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            f      = F.normalize(backbone.encode_image(imgs).float(), dim=-1)
            logits = head(f)
            loss   = crit_b(logits, labels) / ACCUM_STEPS
        scaler.scale(loss).backward()
        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader_b):
            scaler.unscale_(opt_b)
            torch.nn.utils.clip_grad_norm_(
                [p for p in backbone.parameters() if p.requires_grad] + list(head.parameters()), 1.0)
            scaler.step(opt_b); scaler.update()
            opt_b.zero_grad()
            ema_b.update(head)
        tot_loss    += loss.item() * ACCUM_STEPS
        tot_correct += (logits.argmax(1) == labels).sum().item()
        tot_n       += labels.size(0)

    backbone.eval(); head.eval()
    ema_b.apply(head)
    rp = eval_head_on_cached(head.weight, X_val, val_ids, val_queries, val_gt, alpha=0.0)
    ema_b.restore(head)

    print(f'  Epoch {epoch}/{EPOCHS_B}  loss={tot_loss/len(train_loader_b):.4f}  '
          f'acc={tot_correct/tot_n:.4f}  val_R={rp:.4f}')

    if rp > best_rp_b:
        best_rp_b = rp; no_improve = 0
        ema_b.apply(head)
        torch.save({'head': head.state_dict(), 'backbone': backbone.state_dict(), 'epoch': epoch},
                   CKPT_DIR / 'stage_b_best.pt')
        ema_b.restore(head)
        print(f'    → Saved best')
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'  Early stopping at epoch {epoch}'); break

print(f'\nStage B best val R-prec: {best_rp_b:.4f}')


In [ ]:
# ── Load best checkpoint & re-extract features ────────────────────────────────
from src.features import extract_image_features, save_features

ckpt = torch.load(CKPT_DIR / "stage_b_best.pt")
backbone.load_state_dict(ckpt["backbone"])
head.load_state_dict(ckpt["head"])
backbone.eval(); head.eval()
print(f"Loaded Stage B best (epoch {ckpt['epoch']})")

FEAT_DIR_FT = OUT_DIR / "features" / "siglip_so400m_ft"
FEAT_DIR_FT.mkdir(parents=True, exist_ok=True)

for split, df, img_dir in [
    ("val",  val_gallery,  DATA_DIR/"validation_gallery"),
    ("test", test_gallery, DATA_DIR/"test_gallery"),
]:
    ds     = GalleryDataset(df, img_dir, transform=tf_val)
    loader = DataLoader(ds, batch_size=128, shuffle=False,
                         num_workers=4, pin_memory=True)
    feats, ids = extract_image_features(backbone, loader, device=DEVICE)
    save_features(feats, ids, FEAT_DIR_FT, split)

X_val_ft,  val_ids_ft  = load_features(FEAT_DIR_FT, "val",  gallery_df=val_gallery)
X_test_ft, test_ids_ft = load_features(FEAT_DIR_FT, "test", gallery_df=test_gallery)
print("Fine-tuned features extracted")

In [ ]:
# ── WiSE-FT blend sweep: fine-tuned + zero-shot ───────────────────────────────
from src.ensemble import blend_zeroshot_finetuned

W = head.weight.detach().cpu().numpy()   # (K, D)

# Fine-tuned logits
S_ft_val  = temperature_scale(X_val_ft  @ W.T, T=1.0, to_prob=True)
# Zero-shot logits (from cached SigLIP zero-shot features)
S_zs_prob = temperature_scale(X_val @ T_proto.T, T=1.0, to_prob=True)

print("WiSE-FT alpha sweep (fine-tuned vs zero-shot):")
best_alpha, best_rp = 0.0, -1.0
for alpha in np.arange(0.0, 1.05, 0.1):
    S_blend = (1 - alpha) * S_zs_prob + alpha * S_ft_val
    preds   = score_to_preds(S_blend, val_ids, val_queries)
    rp      = evaluate(preds, val_gt)["r_precision"]
    mark    = " ←" if rp > best_rp else ""
    print(f"  alpha={alpha:.1f}  R-prec={rp:.4f}{mark}")
    if rp > best_rp:
        best_rp, best_alpha = rp, float(alpha)

print(f"\nBest alpha: {best_alpha:.1f}  R-prec={best_rp:.4f}")
print(f"Phase 4 reference (WiSE-FT logreg): 0.9287")
print(f"Improvement: {best_rp - 0.9287:+.4f}")

In [ ]:
# ── Full eval of best fine-tuned config ───────────────────────────────────────
S_zs_prob_test = temperature_scale(X_test @ T_proto.T, T=1.0, to_prob=True)
S_ft_test      = temperature_scale(X_test_ft @ W.T,    T=1.0, to_prob=True)
S_final_val    = (1 - best_alpha) * S_zs_prob   + best_alpha * S_ft_val
S_final_test   = (1 - best_alpha) * S_zs_prob_test + best_alpha * S_ft_test

preds   = score_to_preds(S_final_val, val_ids, val_queries)
metrics = evaluate(preds, val_gt)
print(f"Fine-tuned WiSE-FT (α={best_alpha:.1f}) full metrics:")
print_metrics(metrics)

In [ ]:
# ── Build & save submission ───────────────────────────────────────────────────
from src.submit import build_submission, validate_submission
import datetime, subprocess, sys

test_query_ids, test_cat_indices = [], []
for _, row in test_queries.iterrows():
    cat_id, _, _ = matcher.predict(str(row["description"]))
    test_query_ids.append(str(row["query_id"]))
    test_cat_indices.append(cat_id2idx.get(cat_id, 0))

sub_df   = build_submission(S_final_test, test_ids_ft,
                             test_query_ids, test_cat_indices, top=20)
is_valid = validate_submission(sub_df, test_gallery)

ts       = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
tag      = f"siglip_ft_wiseft_a{best_alpha:.1f}"
sub_path = OUT_DIR / "submissions" / f"{ts}_{tag}.csv"
sub_df.to_csv(sub_path, index=False)
sub_df.to_csv(OUT_DIR / "submission.csv", index=False)
print(f"Saved: {sub_path.name}")

r = subprocess.run(
    [sys.executable, str(DATA_DIR/"validate_submission.py"),
     str(OUT_DIR/"submission.csv"), "--data-dir", str(DATA_DIR)],
    capture_output=True, text=True)
print(r.stdout)
if r.stderr: print(r.stderr)
print("Validator exit code:", r.returncode)

today = datetime.date.today()
print(f"""
=== Add to experiment_log.md ===
| E010 | {today} | 5 | SigLIP SO400M fine-tuned + WiSE-FT | StageA={EPOCHS_A}ep + StageB={EPOCHS_B}ep, α={best_alpha:.1f}, seed=42 | 42 | {metrics['r_precision']:.4f} | {metrics['map@20']:.4f} | {metrics['p@5']:.4f} | {metrics['hit@1']:.4f} | {metrics['hit@5']:.4f} | {metrics['mrr@20']:.4f} | phase5 |
""")